In [1]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset

In [2]:
# 加载编码器
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-chinese')

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

In [3]:
tokenizer(
    ['从明天开始做一个幸福的人。', '我太爱了，简直。']
)

{'input_ids': [[101, 794, 3209, 1921, 2458, 1993, 976, 671, 702, 2401, 4886, 4638, 782, 511, 102], [101, 2769, 1922, 4263, 749, 8024, 5042, 4684, 511, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [4]:
class MyDataset(Dataset):
    def __init__(self, split):
        self.dataset = load_dataset('lansinuote/ChnSentiCorp')[split]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        return self.dataset[index]['text'], self.dataset[index]['label']

dataset = MyDataset('train')

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
# 数据整理函数
def collate_fn(batch):
    sents = [
        sent
        for sent, _ in batch
    ]

    labels = [
        label
        for _, label in batch
    ]

    data = tokenizer(
        sents,
        truncation=True,
        padding=True,
        max_length=500,
        return_tensors='pt',
    )

    input_ids = data['input_ids']
    attention_mask = data['attention_mask']
    token_type_ids = data['token_type_ids']

    return {
        'input_ids': input_ids.to(device),
        'attention_mask': attention_mask.to(device),
        'token_type_ids': token_type_ids.to(device),
        'labels': torch.tensor(labels, dtype=torch.long).to(device),
    }

In [7]:
# 初始 collate_fn

collate_fn([('nasdhjwoid', 1), ('sdfewfw', 0)])

{'input_ids': tensor([[  101, 10157,  8168,  8199,  8334, 10911,  8601,   102],
         [  101,  9201,  9568, 12242,  8220,   102,     0,     0]],
        device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1],
         [1, 1, 1, 1, 1, 1, 0, 0]], device='cuda:0'),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'),
 'labels': tensor([1, 0], device='cuda:0')}

In [8]:
dataloader = DataLoader(
    dataset,
    batch_size=16,
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

In [ ]:
# 加载模型
pretrained_model = AutoModel.from_pretrained('google-bert/bert-base-chinese').to(device)

In [ ]:
sum(param.numel() for param in pretrained_model.parameters())